# Team Classification Inference Grid

**Purpose:** Run every team-classification configuration once over the three clips and
record raw per-crop predictions with thresholds disabled, so method comparisons and
threshold selection happen offline on a fixed record.  
**Inputs:** `data/raw/clip_*.mp4`, `data/processed/<clip>/player_detections.pkl`,
`config/default.yaml`.  
**Outputs:** one CSV per run plus `manifest.csv` in `data/outputs/inference_grid/`.  
**Backs:** `results/team_classification/inference_grid/`.

The first cell pins the working directory to the repo root.

In [1]:
import os
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
os.chdir(REPO_ROOT)
print(os.getcwd())

/home/jovyan/nba-video-analytics


## 1. The grid

Sixty runs: five arms (FashionCLIP under three prompt schemes, K-means, and embedding
clustering) by four crop fractions by three clips. Prompt variants are schemes applied
per clip rather than fixed pairs, since clip_3 is white against red. Confidence
thresholds are 0.0 throughout: every raw score is recorded and thresholds are selected
offline. K-means fits are seeded inside the classifiers (random_state=42), so runs are
repeatable.

In [2]:
import sys, pickle, cv2, yaml, itertools, time
import pandas as pd
sys.path.insert(0, '.')

from basketball.team_classifier.classifier import (
    FashionCLIPClassifier, KMeansClassifier, EmbeddingClusteringClassifier
)

with open('config/default.yaml') as f:
    cfg = yaml.safe_load(f)
tc = cfg['team_classifier']

CLIPS = ['clip_1', 'clip_2', 'clip_3']
CROP_FRACTIONS = [0.5, 0.667, 0.85, 1.0]

# Prompt variants are SCHEMES applied per clip, not fixed pairs: clip_3 is
# white vs red, so a fixed pair cannot express the same scheme across clips.
PROMPT_VARIANTS = {
    'A_generic': {
        'clip_1': ('white shirt', 'dark blue shirt'),
        'clip_2': ('white shirt', 'dark blue shirt'),
        'clip_3': ('white shirt', 'red shirt'),
    },
    'B_domain': {
        'clip_1': ('white basketball jersey', 'dark blue basketball jersey'),
        'clip_2': ('white basketball jersey', 'dark blue basketball jersey'),
        'clip_3': ('white basketball jersey', 'red basketball jersey'),
    },
    'C_nba': {
        'clip_1': ('white NBA jersey', 'dark blue NBA jersey'),
        'clip_2': ('white NBA jersey', 'dark blue NBA jersey'),
        'clip_3': ('white NBA jersey', 'red NBA jersey'),
    },
}

OUT_DIR = 'data/outputs/inference_grid'
os.makedirs(OUT_DIR, exist_ok=True)

def load_clip(name):
    frames = []
    cap = cv2.VideoCapture(f'data/raw/{name}.mp4')
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    with open(f'data/processed/{name}/player_detections.pkl', 'rb') as f:
        tracks = pickle.load(f)
    return frames, tracks

data = {c: load_clip(c) for c in CLIPS}
for c in CLIPS:
    print(f'{c}: {len(data[c][0])} frames')

clip_1: 117 frames
clip_2: 174 frames
clip_3: 243 frames


## 2. Run the grid

Each run writes one CSV named for its configuration; a run whose CSV already exists is
skipped, so the cell resumes after an interruption. The manifest records every
(configuration, path) pair. The tokenizer FutureWarning in the output is a known
transformers notice and harmless.

In [3]:
def run_path(method, variant, crop, clip):
    v = f'_{variant}' if variant else ''
    return f'{OUT_DIR}/{method}{v}_crop{crop}_{clip}.csv'

def build(method, variant, crop, clip):
    # Thresholds fixed at 0.0 throughout: every raw pre-threshold score is
    # recorded, and thresholds are selected offline per method per fold.
    if method == 'fashionclip':
        t1, t2 = PROMPT_VARIANTS[variant][clip]
        return FashionCLIPClassifier(
            team_1_description=t1, team_2_description=t2,
            reset_interval=tc['reset_interval'],
            crop_fraction=crop, confidence_threshold=0.0,
        )
    if method == 'kmeans':
        return KMeansClassifier(
            crop_fraction=crop,
            fit_stride=tc['kmeans_fit_stride'],
            bg_distance_threshold=tc['kmeans_background_distance_threshold'],
            margin_threshold=0.0,
        )
    encoder = FashionCLIPClassifier(
        team_1_description='unused', team_2_description='unused',
        reset_interval=tc['reset_interval'],
        crop_fraction=crop, confidence_threshold=0.0,
    )
    return EmbeddingClusteringClassifier(
        encoder=encoder,
        fit_stride=tc['embedding_fit_stride'],
        margin_threshold=0.0,
    )

jobs = []
for crop, clip in itertools.product(CROP_FRACTIONS, CLIPS):
    for variant in PROMPT_VARIANTS:
        jobs.append(('fashionclip', variant, crop, clip))
    jobs.append(('kmeans', None, crop, clip))
    jobs.append(('embedding', None, crop, clip))

print(f'{len(jobs)} runs total')

manifest = []
t_start = time.time()
for i, (method, variant, crop, clip) in enumerate(jobs, 1):
    path = run_path(method, variant, crop, clip)
    manifest.append({'method': method, 'variant': variant or '',
                     'crop_fraction': crop, 'clip': clip, 'path': path})
    if os.path.exists(path):
        print(f'[{i}/{len(jobs)}] skip (exists) {path}')
        continue
    frames, tracks = data[clip]
    clf = build(method, variant, crop, clip)
    t0 = time.time()
    clf.assign_teams(frames, tracks, cache_path=None,
                     record_path=path, clip_name=clip)
    print(f'[{i}/{len(jobs)}] {method} {variant or "-"} crop={crop} {clip} '
          f'({time.time()-t0:.1f}s)')

pd.DataFrame(manifest).to_csv(f'{OUT_DIR}/manifest.csv', index=False)
print(f'\nDone in {(time.time()-t_start)/60:.1f} min. Manifest written.')

60 runs total
[1/60] skip (exists) data/outputs/inference_grid/fashionclip_A_generic_crop0.5_clip_1.csv
[2/60] skip (exists) data/outputs/inference_grid/fashionclip_B_domain_crop0.5_clip_1.csv
[3/60] skip (exists) data/outputs/inference_grid/fashionclip_C_nba_crop0.5_clip_1.csv
[4/60] skip (exists) data/outputs/inference_grid/kmeans_crop0.5_clip_1.csv
[5/60] skip (exists) data/outputs/inference_grid/embedding_crop0.5_clip_1.csv
[6/60] skip (exists) data/outputs/inference_grid/fashionclip_A_generic_crop0.5_clip_2.csv
[7/60] skip (exists) data/outputs/inference_grid/fashionclip_B_domain_crop0.5_clip_2.csv
[8/60] skip (exists) data/outputs/inference_grid/fashionclip_C_nba_crop0.5_clip_2.csv
[9/60] skip (exists) data/outputs/inference_grid/kmeans_crop0.5_clip_2.csv
[10/60] skip (exists) data/outputs/inference_grid/embedding_crop0.5_clip_2.csv
[11/60] skip (exists) data/outputs/inference_grid/fashionclip_A_generic_crop0.5_clip_3.csv
[12/60] skip (exists) data/outputs/inference_grid/fashionc

## 3. Sanity summary

Checks every manifest path exists and that row counts per clip are identical across
all runs of a clip, then summarises median confidence and unusable-crop rate by method
and crop fraction.

In [4]:
man = pd.read_csv(f'{OUT_DIR}/manifest.csv')
print(f'{len(man)} runs in manifest')
missing = [p for p in man['path'] if not os.path.exists(p)]
print(f'missing files: {len(missing)}')

rows = []
for _, r in man.iterrows():
    df = pd.read_csv(r['path'])
    rows.append({**r.to_dict(), 'n': len(df),
                 'unusable_pct': round((~df['crop_ok']).mean()*100, 2),
                 'conf_med': round(df['confidence'].median(), 3)})
summary = pd.DataFrame(rows)

print('\nRow counts per clip (must be identical across all runs of a clip):')
print(summary.groupby('clip')['n'].agg(['min', 'max', 'nunique']))

print('\nMedian confidence by method and crop fraction:')
print(summary.pivot_table(index='method', columns='crop_fraction',
                          values='conf_med', aggfunc='median'))

print('\nUnusable-crop % by method and crop fraction:')
print(summary.pivot_table(index='method', columns='crop_fraction',
                          values='unusable_pct', aggfunc='max'))

60 runs in manifest
missing files: 0



Row counts per clip (must be identical across all runs of a clip):
         min   max  nunique
clip                       
clip_1   728   728        1
clip_2  1623  1623        1
clip_3  1887  1887        1

Median confidence by method and crop fraction:
crop_fraction  0.500  0.667  0.850  1.000
method                                   
embedding      0.578  0.584  0.580  0.578
fashionclip    0.991  0.993  0.994  0.995
kmeans         0.794  0.791  0.768  0.776

Unusable-crop % by method and crop fraction:
crop_fraction  0.500  0.667  0.850  1.000
method                                   
embedding        0.0    0.0    0.0    0.0
fashionclip      0.0    0.0    0.0    0.0
kmeans           0.0    0.0    0.0    0.0


## 4. Outcome

Sixty run CSVs and `manifest.csv` are written to `data/outputs/inference_grid/`. The
copies shipped with the repository are in
`results/team_classification/inference_grid/`;
`scripts/team_classification_sweep.ipynb` scores them.